# Retail Sales & KPI Performance Analytics (Superstore EDA)

This notebook cleans the raw Superstore transactions dataset, loads a normalized Star Schema into DuckDB, runs analytical SQL window functions across 4 regions and 3 categories, isolates Furniture discount leakage (`Tables` and `Bookcases`), and simulates margin recovery.

In [1]:
import pandas as pd
from retail_kpi.clean import clean_superstore, build_star_schema_duckdb
from retail_kpi.analyze import run_sql_queries, simulate_pricing_recommendations

df, stats = clean_superstore()
print('Cleaning summary:', stats)
db_path = build_star_schema_duckdb(df)
queries = run_sql_queries(db_path)
df.head(3)

Cleaning summary: {'raw_rows': 10800, 'dropped_incomplete_rows': 806, 'clean_rows': 9994}


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Sales,Quantity,Discount,Profit,order_year,order_quarter,order_month,ship_days,profit_margin,discount_band
0,1,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,261.96,2,0.0,41.9136,2017,2017Q4,2017-11,3,0.16,0% (Full Price)
1,2,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,731.94,3,0.0,219.5820,2017,2017Q4,2017-11,3,0.30,0% (Full Price)
2,3,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,14.62,2,0.0,6.8714,2017,2017Q2,2017-06,4,0.47,0% (Full Price)


In [2]:
print('=== Region x Category Window-Function Summary ===')
display(queries['region_category'])

=== Region x Category Window-Function Summary ===


,region,category,order_volume,line_items,revenue,profit,profit_margin_pct,avg_discount_pct,regional_revenue_share_pct,regional_margin_rank
0,Central,Furniture,403,481,163797.16,-2871.05,-1.75,29.74,32.68,3
1,Central,Office Supplies,880,1422,167026.42,8879.98,5.32,25.27,33.32,2
2,Central,Technology,356,420,170416.31,33697.43,19.77,13.31,34.00,1
3,East,Furniture,488,601,208291.20,3046.17,1.46,15.41,30.69,3
4,East,Technology,443,535,264973.98,47462.04,17.91,14.34,39.04,2
5,East,Office Supplies,1074,1712,205516.05,41014.58,19.96,14.29,30.28,1
6,South,Furniture,278,332,117298.68,6771.21,5.77,12.15,29.94,3
7,South,Technology,255,293,148771.91,19991.83,13.44,10.78,37.98,2
8,South,Office Supplies,619,995,125651.31,19986.39,15.91,16.74,32.08,1
9,West,Furniture,595,707,252612.74,11504.95,4.55,13.14,34.82,3


In [3]:
print('=== Year-over-Year Growth (LAG Window Function) ===')
display(queries['yoy_growth'])

=== Year-over-Year Growth (LAG Window Function) ===


,order_year,order_volume,revenue,profit,profit_margin_pct,avg_order_value,yoy_revenue_growth_pct
0,2015,969,484247.50,49543.97,10.23,499.74,NaN
1,2016,1038,470532.51,61618.60,13.10,453.31,-2.83
2,2017,1315,609205.60,81795.17,13.43,463.27,29.47
3,2018,1687,733215.26,93439.27,12.74,434.63,20.36


In [4]:
print('=== Sub-Category Profit & Discount Ranking ===')
display(queries['subcategory_ranking'].head(8))

=== Sub-Category Profit & Discount Ranking ===


,category,sub_category,order_volume,revenue,profit,profit_margin_pct,avg_discount_pct,loss_severity_rank
0,Furniture,Tables,307,206965.53,-17725.48,-8.56,26.13,1
1,Furniture,Bookcases,224,114880.00,-3472.56,-3.02,21.11,2
2,Office Supplies,Supplies,187,46673.54,-1189.10,-2.55,7.68,3
3,Office Supplies,Fasteners,215,3024.28,949.52,31.40,8.20,4
4,Technology,Machines,112,189238.63,3384.76,1.79,30.61,5
5,Office Supplies,Labels,346,12486.31,5546.25,44.42,6.87,6
6,Office Supplies,Art,731,27118.79,6527.79,24.07,7.49,7
7,Office Supplies,Envelopes,249,16476.40,6964.18,42.27,8.03,8


In [5]:
sim = simulate_pricing_recommendations(df)
for rec in sim['recommendations']:
    print(f"- {rec['title']}: {rec['estimated_impact']}")
print(f"Conservative Profit Recovery: +${sim['combined_conservative_profit_recovery_usd']:,.2f} ({sim['estimated_overall_margin_recovery_pct']:.2f}% relative margin lift)")

- Cap Tables & Bookcases Discounts at 15%: +$55,649 gross profit uplift (+$36,172 at 65% elasticity, ~12.6% overall margin recovery; Furniture margin 2.49% -> 7.36%).
- Require Manager Approval Above 20% Discount Floor: Up to +$204,444 additional gross margin upside if deep clearance discounts >30% are capped at 20%.
- Shift Furniture Incentives to Volume Bundling & Standard Shipping: Protects average order value ($458.61) while eliminating unprofitable sub-20% discount bands.
Conservative Profit Recovery: +$36,172.08 (12.63% relative margin lift)
